In [2]:
from sentence_transformers import SentenceTransformer
import pandas as pd 
import numpy as np
import re 

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Loading model completed")
folder = "Data\\"
categories = pd.read_csv(folder + "amazon_final_category_mapping.csv")
categories["normalized_name"] = categories["category_name"].apply(normalize)
category_names = categories["normalized_name"].tolist()


categories["normalized_name"] = categories["category_name"].apply(normalize)
category_names = categories["normalized_name"].tolist()

category_embeddings = model.encode(
    categories["category_name"].tolist(),
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading model completed


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [44]:
def get_top_categories(llm_category, top_k=3):

    query_embedding = model.encode(
        [llm_category],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0].astype(np.float32)

    embedding_scores = category_embeddings @ query_embedding

    top_indices = np.argsort(-embedding_scores)[:top_k]

    results = []
    category_ids=[]
    print(f"Top {top_k} categories:")
    print("Category_id          Category_name")
    for idx in top_indices:
        print(int(categories.iloc[idx]["category_id"]),categories.iloc[idx]["category_name"])
        results.append({
            "category_id": int(categories.iloc[idx]["category_id"]),
            "category_name": categories.iloc[idx]["category_name"],
            "embedding_score": round(float(embedding_scores[idx]), 4),
        }) 
        category_ids.append( int(categories.iloc[idx]["category_id"]))
    
    return results,category_ids

In [4]:
from src.Search.Data_ingest import data_ingest 

products,categories = data_ingest()

Ingesting Data
Procesisng Data
Columns:
['Unnamed: 0', 'asin', 'title', 'category_name', 'category_id', 'stars', 'reviews', 'price', 'price_inr', 'imgUrl', 'productURL']

Number of products: 1393565


In [5]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
client_2=Groq(api_key=os.getenv("GROQ_API_KEY_2"))

# Models list for further changing 
# models = client.models.list()
# for model in models.data:
#     print(model.id)


In [19]:
from src.LLM.TEMPLATES.main_template import PROMPT_TEMPLATE 
from src.LLM.TEMPLATES.min_attribute import PROMPT_TEMPLATE_ATT_MIN 
from src.LLM.TEMPLATES.product_template import PROMPT_TEMPLATE_PRODUCT

In [ ]:
import json 

def attributes(template,promt_main):
    att_template={}
    att_template["user_promt"]=promt_main 
    att_template["items"]=[]
    for i,values in enumerate(template['items']):
        att_template["items"].append({})
        att_template["items"][i]['name']=values['name']
        att_template["items"][i]['preferences']=values['preferences']

    prompt =  PROMPT_TEMPLATE_ATT_MIN.replace(
        "{input_json}",
        json.dumps(att_template, ensure_ascii=False)
    )
    response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "system",
                    "content": "Extract attributes accurately. Return JSON only."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
    )
    att_template=json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    for i,values in enumerate(template['items']):
        template["items"][i]['attributes']=att_template["items"][i]['attributes']
    return template

def elaborate_product(product_name,preferences):
    input_data = {"item": product_name,"preferences":preferences}

    prompt = PROMPT_TEMPLATE_PRODUCT.replace(
        "{input_json}",
        json.dumps(input_data, ensure_ascii=False)
    )

    response = client_2.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Elaborate product names for semantic search. "
                    "Return JSON only."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    print("Elaborate Promt:",result["output"])
    return result["output"]
     

    
def adding_category(template):
    llm_items=template["items"]
    for i in range(len(llm_items)):
            llm_category=template["items"][i]["name"]
            preferences=template["items"][i]["preferences"]
            elaborate_promt=elaborate_product(llm_category,preferences)
            results,category_ids=get_top_categories(elaborate_promt)
            template["items"][i]["category_id"]=category_ids 
    return template

def extract_requirements(user_prompt):
    prompt = PROMPT_TEMPLATE.replace("{user_prompt}", user_prompt)

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": "Extract requirements accurately. Return JSON only."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    template=json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    template=attributes(template,user_prompt)
    main_template=adding_category(template)
    
    
    return main_template

In [ ]:
prompt="Show me one black shirt for males and one blue shirt for females"
main_template=extract_requirements(prompt)
print(main_template)

Elaborate Promt: A black men's shirt is a garment worn on the upper body for casual or formal occasions.
Top 5 categories:
Category_id          Category_name
110 Men's Clothing
84 Boys' Clothing
112 Men's Accessories
116 Women's Clothing
114 Men's Shoes
Elaborate Promt: A shirt is a women's upper-body garment, often blue, worn for casual or formal occasions.
Top 5 categories:
Category_id          Category_name
116 Women's Clothing
110 Men's Clothing
91 Girls' Clothing
120 Women's Accessories
68 Wearable Technology
{'items': [{'name': 'Shirts', 'keywords': ['shirt'], 'preferences': ['black', 'men'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['black'], 'target_user': ['men']}, 'category_id': [110, 84, 112, 116, 114]}, {'name': 'Shirts', 'keywords': ['shirt'], 'preferences': ['blue', 'women'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['blue'], 'target_user': ['women']}, 'category_id': [116, 110, 91, 120, 68]}]}

In [21]:
main_template

{'items': [{'name': 'Shirts',
   'keywords': ['shirt'],
   'preferences': ['black', 'men'],
   'budget': None,
   'min_rating': None,
   'min_reviews': None,
   'attributes': {'color': ['black'], 'target_user': ['men']},
   'category_id': [110, 84, 112, 116, 114]},
  {'name': 'Shirts',
   'keywords': ['shirt'],
   'preferences': ['blue', 'women'],
   'budget': None,
   'min_rating': None,
   'min_reviews': None,
   'attributes': {'color': ['blue'], 'target_user': ['women']},
   'category_id': [116, 110, 91, 120, 68]}]}

In [48]:
from src.Search.product_search import search_products
import pandas as pd

class Searching:
    def __init__(self,products,categories):
        print("Getting all the Data")
        self.products, self.categories = products,categories
        self.nodes_explored = 0
        self.backtracks = 0

    def get_domains(self, items):
        domains = {}
        total_possible_combinations = 1

        for item_name, item in items.items():
            category_ids = item["category_id"]

            if not isinstance(category_ids, list):
                category_ids = [category_ids]

            products = self.products

            domain = products[
                products["asin"].notna() &
                products["title"].notna() &
                products["category_id"].isin(category_ids) &
                products["category_name"].notna() &
                products["price_inr"].notna() &
                products["stars"].notna() &
                products["reviews"].notna()
            ]

            if item.get("budget") is not None:
                domain = domain[domain["price_inr"] <= item["budget"]]

            if item.get("min_rating") is not None:
                domain = domain[domain["stars"] >= item["min_rating"]]

            if item.get("min_reviews") is not None:
                domain = domain[domain["reviews"] >= item["min_reviews"]]

            domain = search_products(domain, item)

            domains[item_name] = domain
            total_possible_combinations *= len(domain)

            print(item_name, "domain size:", len(domain))

        return domains, total_possible_combinations

    def is_consistent(self, assignment, new_product, item, requirements):
        for product in assignment.values():
            if product["asin"] == new_product["asin"]:
                return False

        if item.get("budget") is not None:
            if new_product["price_inr"] > item["budget"]:
                return False

        if item.get("min_rating") is not None:
            if new_product["stars"] < item["min_rating"]:
                return False

        if item.get("min_reviews") is not None:
            if new_product["reviews"] < item["min_reviews"]:
                return False

        total_cost = new_product["price_inr"]
        for product in assignment.values():
            total_cost += product["price_inr"]

        if requirements.get("budget") is not None:
            if total_cost > requirements["budget"]:
                return False

        return True

   
    def backtracking_search(
        self, variables, domains, assignment, requirements, items,
        solutions, max_solutions
    ):
        if len(solutions) >= max_solutions:
            return

        if len(assignment) == len(variables):
            solutions.append(assignment.copy())
            return

        variable = variables[len(assignment)]
        item = items[variable]

        for index in domains[variable].index:
            if len(solutions) >= max_solutions:
                return

            self.nodes_explored += 1
            product = self.products.loc[index]

            if self.is_consistent(
                assignment, product, item, requirements
            ):
                assignment[variable] = product

                self.backtracking_search(
                    variables, domains, assignment, requirements, items,
                    solutions, max_solutions
                )

                del assignment[variable]


    def finding_products(self, requirements):
        print(requirements)

        # variables = [item["name"] for item in requirements["items"]]
        # items = {item["name"]: item for item in requirements["items"]}
        from collections import defaultdict

        variables = []
        items = {}
        name_count = defaultdict(int)

        for item in requirements["items"]:
            name = item["name"].strip().lower()
            name_count[name] += 1

            unique_name = f"{name}-{name_count[name]}"

            variables.append(unique_name)
            items[unique_name] = item

        self.nodes_explored = 0
        self.backtracks = 0

        domains, total_possible_combinations = self.get_domains(items)
        print("Total Possible Combiantions:",total_possible_combinations)
        top_k_solutions=10

        # Fail early if any item has an empty domain
        if any(domains[v].empty for v in variables):
            print("NO FEASIBLE SOLUTION: One or more domains are empty.")
            return None

        solutions = []

        self.backtracking_search(
            variables, domains, {}, requirements, items,
            solutions, max_solutions=top_k_solutions
        )

        if not solutions:
            print("NO FEASIBLE SOLUTION FOUND")
        else:
            for i, solution in enumerate(solutions, 1):
                print(f"\nSolution {i}")

                total_cost = 0

                for variable, product in solution.items():
                    print("Item:", variable)
                    print("Product:", product["title"])
                    print("Price:", product["price_inr"])
                    print("Category ID:", product["category_id"])
                    print("Category Name:", product["category_name"])
                    total_cost += product["price_inr"]
                    print("------------------------")
                    
                print("=============================")
                print("Total cost:", total_cost)


        print("Budget:", requirements.get("budget"))
        print("Nodes explored:", self.nodes_explored)
        print("Backtracks:", self.backtracks)

        return solutions


In [49]:

obj=Searching(products,categories)

Getting all the Data


In [50]:
# prompt="show me black hiking shoes under 50 with white t-shirt with  L size with black shorts with L size"
# prompt=str(input("Enter the prompt"))
prompt="Show me one black shirt for males and one blue shoes for females"
print(prompt)
main_template=extract_requirements(prompt)
print(main_template)
solutions=obj.finding_products(main_template)

Show me one black shirt for males and one blue shoes for females
Elaborate Promt: A black shirt for men is a garment worn on the upper body, typically with a collar and sleeves, suitable for casual or formal attire.
Top 3 categories:
Category_id          Category_name
110 Men's Clothing
84 Boys' Clothing
116 Women's Clothing
Elaborate Promt: Blue women's shoes are footwear designed for women to protect and support the feet during walking, standing, and everyday activities.
Top 3 categories:
Category_id          Category_name
122 Women's Shoes
97 Girls' Shoes
114 Men's Shoes
{'items': [{'name': 'Shirts', 'keywords': ['shirt'], 'preferences': ['black', 'men'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['black'], 'target_user': ['men']}, 'category_id': [110, 84, 116]}, {'name': 'Shoes', 'keywords': ['shoes'], 'preferences': ['blue', 'women'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['blue'], 'target_user': ['w